In [2]:
import duckdb as db

In [3]:
db.sql("SELECT 'Hello, DuckDB!' AS greeting").df()

,greeting
0,"Hello, DuckDB!"


In [4]:
sql = db.sql("SELECT * FROM range(10) AS nums")

In [5]:
QUERY = """
INSTALL spatial;
LOAD spatial; 

SELECT
    ST_GeomFromText('POLYGON((0 0, 0 2, 2 3, 3 0, 0 0))') as g,
    ST_AsText(ST_GeomFromText('POLYGON((0 0, 0 2, 2 3, 3 0, 0 0))')) AS geometry_text,
    ST_Within(ST_Centroid(g), g) as centroid_inside
"""
# db.sql(QUERY).df()

db.sql(
    "DESCRIBE SELECT * FROM read_parquet('s3://overturemaps-us-west-2/release/2026-08-19.0/theme=buildings/type=building/*.parquet')"
).df()

,column_name,column_type,null,key,default,extra
0,id,VARCHAR,YES,None,None,None
1,names,"STRUCT(""primary"" VARCHAR, common MAP(VARCHAR, ...",YES,None,None,None
2,sources,"STRUCT(property VARCHAR, dataset VARCHAR, lice...",YES,None,None,None
3,level,INTEGER,YES,None,None,None
4,height,DOUBLE,YES,None,None,None
5,min_height,DOUBLE,YES,None,None,None
6,is_underground,BOOLEAN,YES,None,None,None
7,num_floors,INTEGER,YES,None,None,None
8,num_floors_underground,INTEGER,YES,None,None,None
9,min_floor,INTEGER,YES,None,None,None


In [11]:
# to check which centroid strategy we shall use
# EPSG:4326
BBOX = {"xmin": -103.473186, "ymin": 44.703741, "xmax": -103.382807, "ymax": 44.740576}

QUERY = f"""
INSTALL spatial;
LOAD spatial; 

WITH src AS (
    SELECT id, geometry
    FROM read_parquet('s3://overturemaps-us-west-2/release/2026-08-19.0/theme=buildings/type=building/*.parquet')
    WHERE 
        bbox.xmin >= {BBOX["xmin"]} AND bbox.xmax <= {BBOX["xmax"]} AND
        bbox.ymin >= {BBOX["ymin"]} AND bbox.ymax <= {BBOX["ymax"]}
),
proj AS (
    SELECT id, ST_Transform(geometry, 'EPSG:4326', 'EPSG:5070', always_xy := true) AS g
    FROM src
)
SELECT 
    id,
    ST_GeometryType(g) as gtype,
    ST_IsValid(g) as is_valid,
    ST_Within(ST_Centroid(g), g) as centroid_inside,
    ST_Distance(ST_Centroid(g), ST_PointOnSurface(g)) as offset_m,
    ST_Area(g) as area_m2
FROM proj
LIMIT 10;
"""

df = db.sql(QUERY).df()

In [12]:
df

,id,gtype,is_valid,centroid_inside,offset_m,area_m2
0,d00bbbec-8d07-4d98-91e3-1e05366953e0,POLYGON,True,True,3.024676e+00,121.260877
1,76627a46-e41c-43c2-9f52-f841819abde1,POLYGON,True,True,7.983396e-07,72.986591
2,3014e6ed-383d-42e0-91e2-d91a3f0629d8,POLYGON,True,True,8.826543e-04,87.803376
3,bafb08ec-094a-47ff-87cb-ea3f41e3be2b,POLYGON,True,True,2.312981e-01,150.749309
4,877816f8-1366-42a9-9843-99b7102d405e,POLYGON,True,True,2.406851e+00,157.066923
5,ac94cb94-b03d-43ff-9c9c-e40b67ed3004,POLYGON,True,True,6.482780e-04,26.023008
6,bde69b78-d95a-4abd-b33e-7033829d3180,POLYGON,True,True,7.985107e-07,42.557002
7,42da9158-f663-4be5-8693-f017231f7632,POLYGON,True,True,3.188221e-03,230.372027
8,36a7fc6d-a442-4c0b-9ca1-a12e9bf10639,POLYGON,True,True,1.714177e-06,154.166370
9,995a6c53-b237-4b77-ab84-01a5b4470918,POLYGON,True,True,1.158553e-06,93.796597


In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   id               10 non-null     str     
 1   gtype            10 non-null     category
 2   is_valid         10 non-null     bool    
 3   centroid_inside  10 non-null     bool    
 4   offset_m         10 non-null     float64 
 5   area_m2          10 non-null     float64 
dtypes: bool(2), category(1), float64(2), str(1)
memory usage: 766.0 bytes


In [14]:
df["is_valid"].value_counts()

is_valid
True    10
Name: count, dtype: int64